In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install sentence-transformers
!pip install pandas

In [3]:
from pathlib import Path

import pandas as pd

from sentence_transformers import SentenceTransformer

In [4]:
PROJECT_PATH = Path("/content/drive/MyDrive/uterine-emg-rag")

PROCESSED_PATH = PROJECT_PATH / "processed"

In [5]:
chunks_df = pd.read_csv(PROCESSED_PATH / "chunks.csv")

chunks_df.head()

,paper,chunk_id,text,characters
0,paper1,0,Contents lists available at ScienceDirect\nArt...,927
1,paper1,1,Keywords:\nUterine electromyography\nUterine a...,946
2,paper1,2,"transform, and for data classification, such a...",996
3,paper1,3,3\n2.1. \nUterine contraction classification ....,831
4,paper1,4,6\n3.1.1. \nSensing electrodes ..................,966


In [6]:
len(chunks_df)

833

In [7]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
sample = chunks_df.iloc[0]["text"]

embedding = model.encode(sample)

In [9]:
print(type(embedding))

<class 'numpy.ndarray'>


In [10]:
len(embedding)

384

In [11]:
embedding[:10]

array([-0.02798636, -0.07215511, -0.01450806, -0.04743121, -0.02465316,
       -0.00892437, -0.06463538,  0.04219696, -0.00115886,  0.04200175],
      dtype=float32)

In [12]:
texts = chunks_df["text"].tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

In [13]:
embeddings.shape

(833, 384)

In [14]:
import numpy as np

EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
EMBEDDINGS_PATH.mkdir(exist_ok=True)

np.save(
    EMBEDDINGS_PATH / "embeddings.npy",
    embeddings
)

In [15]:
chunks_df.to_csv(
    EMBEDDINGS_PATH / "chunk_metadata.csv",
    index=False
)

In [16]:
loaded = np.load(
    EMBEDDINGS_PATH / "embeddings.npy"
)

loaded.shape

(833, 384)